# 雾密度感知网络：最简完整实验流程

本 notebook 只保留完成一次完整实验所必需的步骤：

`环境初始化 → 数据准备 → 生成数据划分 → 正式训练 → 加载 best.pth 可视化`

每个部分都可以重复运行，但数据准备和 split 生成在数据不变时只需要做一次。测试审计和完整指标评估放在最后，作为可选步骤。

## 1. 环境初始化

作用：挂载 Google Drive、获取项目代码、安装依赖、进入项目目录并检查 GPU。

新建 Colab 运行时需要运行一次；同一个运行时不要反复 clone 项目。

In [ ]:
from pathlib import Path
import sys
import subprocess

PROJECT_ROOT = Path('/content/CrossModal-Defog-OVOD')

if not PROJECT_ROOT.exists():
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run(
        ['git', 'clone', 'https://github.com/codeGease-1/CrossModal-Defog-OVOD.git', str(PROJECT_ROOT)],
        check=True,
    )
else:
    print(f'项目目录已存在，跳过 clone：{PROJECT_ROOT}')

%cd /content/CrossModal-Defog-OVOD
%pip install -q -r requirements.txt

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. 数据准备

作用：把 RSHaze+ 放到项目要求的目录 `datasets/RSHaze+/`。

推荐先把 `RSHaze+.zip` 放在 Google Drive 的 `MyDrive/dataset/` 下。若项目目录中已经有完整数据集，这个 cell 会自动跳过解压。

不要删除数据集目录；删除会导致后续训练重新准备数据。

In [ ]:
from pathlib import Path
import shutil

PROJECT_ROOT = Path('/content/CrossModal-Defog-OVOD')
DATASETS_DIR = PROJECT_ROOT / 'datasets'
DATASET_ROOT = DATASETS_DIR / 'RSHaze+'
DRIVE_ZIP = Path('/content/drive/MyDrive/dataset/RSHaze+.zip')

if DATASET_ROOT.exists():
    print(f'数据集已存在，跳过解压：{DATASET_ROOT}')
else:
    if not DRIVE_ZIP.exists():
        raise FileNotFoundError(
            f'找不到 {DRIVE_ZIP}。请先将 RSHaze+.zip 上传到该 Drive 路径，或手动将数据集放到 {DATASET_ROOT}。'
        )
    DATASETS_DIR.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(DRIVE_ZIP, DATASETS_DIR)

    extracted_candidates = [
        DATASETS_DIR / 'RSHaze+',
        DATASETS_DIR / 'RSHazePlus' / 'RSHaze+',
    ]
    extracted = next((p for p in extracted_candidates if p.exists()), None)
    if extracted is None:
        raise FileNotFoundError('压缩包解压后没有找到 RSHaze+ 目录，请检查压缩包结构。')
    if extracted != DATASET_ROOT:
        shutil.copytree(extracted, DATASET_ROOT, dirs_exist_ok=True)

print(f'数据集目录：{DATASET_ROOT}')
for subset in ['RSHaze_G', 'RSHaze_L', 'RSHaze_S']:
    subset_path = DATASET_ROOT / subset
    if not subset_path.exists():
        raise FileNotFoundError(f'缺少子集目录：{subset_path}')
    print(f'{subset}: OK')

## 3. 生成固定的数据划分

作用：按照项目约定生成固定的 train / val / test 划分，避免每次训练的验证集发生变化。

数据集没有变化时只需要运行一次；如果 `rshazeplus_split.json` 已存在且不需要重新划分，可以跳过这个 cell。

In [ ]:
import os

split_file = 'experiments/haze_density/rshazeplus_split.json'
if os.path.exists(split_file):
    print(f'split 文件已存在，跳过重新生成：{split_file}')
else:
    !python scripts/data/generate_rshazeplus_split.py

!python scripts/data/verify_rshazeplus_split.py

## 4. 正式训练

作用：训练雾密度感知网络，并在验证集表现最好时保存 `best.pth`。

只运行下面一个训练 cell，不要同时运行多个训练命令。第一次训练将 `RESUME_CHECKPOINT = None`；训练中断后，把它改成 `latest.pth` 再重新运行本 cell。

In [ ]:
# 训练参数：第一次训练时 RESUME_CHECKPOINT 保持 None。
EPOCHS = 100
BATCH_SIZE = 4
LEARNING_RATE = 1e-4
RESUME_CHECKPOINT = None

print(f'Epochs={EPOCHS}, batch_size={BATCH_SIZE}, lr={LEARNING_RATE}')

# 使用 !python 直接运行脚本，Colab 会实时显示每一轮 train_loss / val_loss。
if RESUME_CHECKPOINT:
    !python scripts/train/train_haze_density.py --epochs $EPOCHS --batch_size $BATCH_SIZE --lr $LEARNING_RATE --resume "$RESUME_CHECKPOINT"
else:
    !python scripts/train/train_haze_density.py --epochs $EPOCHS --batch_size $BATCH_SIZE --lr $LEARNING_RATE

from pathlib import Path
best_checkpoint = Path('experiments/haze_density/checkpoints/formal/best.pth')
if not best_checkpoint.exists():
    raise FileNotFoundError(f'训练结束但没有找到 {best_checkpoint}')
print(f'训练完成：{best_checkpoint}')

## 5. 加载 best.pth 并展示训练结果

作用：不重新训练模型，直接从测试集抽取样本，显示四联图：

`含雾图像 | 物理先验 | 网络预测雾密度 | 绝对误差`

图像和汇总信息会保存到 `experiments/haze_density/results/notebook_visualization/`，可直接用于汇报。

In [ ]:
from collections import defaultdict
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch

from src.data import build_rshazeplus_dataloader
from src.models.haze_density import HazeDensityNet
from src.models.haze_density.physical_prior import PhysicalPriorModule

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint_path = Path('experiments/haze_density/checkpoints/formal/best.pth')
checkpoint = torch.load(checkpoint_path, map_location='cpu')
config = checkpoint.get('config', {})
model = HazeDensityNet(
    base_channels=config.get('base_channels', 32),
    use_sigmoid=config.get('use_sigmoid', True),
).to(device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

physical_prior = PhysicalPriorModule().to(device).eval()
test_loader = build_rshazeplus_dataloader(
    root='datasets/RSHaze+',
    split='test',
    image_size=256,
    batch_size=4,
    num_workers=0,
    pin_memory=False,
    split_file='experiments/haze_density/rshazeplus_split.json',
)

output_dir = Path('experiments/haze_density/results/notebook_visualization')
output_dir.mkdir(parents=True, exist_ok=True)
max_per_subset = 2
saved = defaultdict(int)

with torch.inference_mode():
    for batch in test_loader:
        images = batch['image'].to(device)
        predictions = model(images).clamp(0, 1)
        priors = physical_prior(images).clamp(0, 1)

        for i, subset in enumerate(batch['subset']):
            if saved[subset] >= max_per_subset:
                continue
            image = images[i].cpu().permute(1, 2, 0).numpy()
            prior = priors[i, 0].cpu().numpy()
            prediction = predictions[i, 0].cpu().numpy()
            error = np.abs(prior - prediction)

            fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
            axes[0].imshow(image); axes[0].set_title('Hazy image')
            axes[1].imshow(prior, cmap='magma', vmin=0, vmax=1); axes[1].set_title('Physical prior')
            axes[2].imshow(prediction, cmap='magma', vmin=0, vmax=1); axes[2].set_title('Predicted density')
            axes[3].imshow(error, cmap='viridis', vmin=0, vmax=1); axes[3].set_title(f'Absolute error (MAE={error.mean():.4f})')
            for axis in axes:
                axis.axis('off')
            fig.suptitle(f'{subset} / {batch["filename"][i]}')
            fig.tight_layout()

            output_path = output_dir / f'{subset}_{Path(batch["filename"][i]).stem}_density_result.png'
            fig.savefig(output_path, dpi=160, bbox_inches='tight')
            plt.show()
            plt.close(fig)
            saved[subset] += 1

        if all(saved[s] >= max_per_subset for s in ['RSHaze_G', 'RSHaze_L', 'RSHaze_S']):
            break

print(f'Device: {device}')
print(f'Checkpoint: {checkpoint_path}')
print(f'保存目录: {output_dir.resolve()}')
print(f'每个子集保存数量: {dict(saved)}')

## 6. 可选：测试集定量评估

如果只需要看图，到上一节就结束。如果需要 MAE、RMSE、Pearson 和更多测试集可视化，再运行下面的 cell。

In [ ]:
# 评估脚本同样使用 !python，指标和文件生成日志会直接显示在 notebook 中。
!python scripts/evaluate/evaluate_haze_density.py \
    --checkpoint experiments/haze_density/checkpoints/formal/best.pth \
    --image_size 256 \
    --batch_size 4 \
    --num_samples_per_subset 16

## 运行规则总结

- 新建 Colab 运行时：运行第 1、2、3、4、5 节。
- 同一运行时重复改参数实验：只重新运行第 4 节，然后运行第 5 节。
- 训练中断：第 4 节把 `RESUME_CHECKPOINT` 改成 `latest.pth` 后重新运行。
- 数据集不变时：不需要反复解压数据，也不需要重新生成 split。
- 第 6 节只在需要定量指标时运行。